In [43]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from transformers import pipeline
import numpy as np
import cv2
import os

#### functions

In [ ]:
class PanoramaClusterer:
    def __init__(self, global_width_px=4096, crop_fov_deg=45):
        """
        global_width_px: The width of the full 360 equirectangular image.
        crop_fov_deg: The Field of View of your crops (e.g., 45 degrees).
        """
        self.global_width_px = global_width_px
        self.crop_fov_deg = crop_fov_deg
        
        # Calculate pixels per degree
        self.px_per_deg = global_width_px / 360.0
        self.crop_width_px = int(self.crop_fov_deg * self.px_per_deg)

    def map_local_to_global(self, image_center_deg, local_x_px):
        """
        Maps a local pixel coordinate (0 to crop_width) to a global angle (0-360).
        """
        # 1. Calculate offset from the center of the crop in pixels
        # Assuming local_x_px is 0-indexed relative to the crop
        crop_center_px = self.crop_width_px / 2.0
        offset_px = local_x_px - crop_center_px
        
        # 2. Convert pixel offset to degrees
        offset_deg = offset_px / self.px_per_deg
        
        # 3. Add to the image center angle and handle wrapping (modulo 360)
        global_angle = (image_center_deg + offset_deg) % 360
        return global_angle

    def cluster_points(self, data_points, k=3):
        """
        Clusters angular data by converting to Unit Vectors (sin, cos).
        """
        angles = np.array([d['global_angle'] for d in data_points])
        
        # Convert degrees to radians
        rads = np.deg2rad(angles)
        
        # Convert to Unit Vectors (Cartesian coordinates)
        # This solves the 0/360 boundary issue for K-Means
        X = np.column_stack([np.cos(rads), np.sin(rads)])
        
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        
        # Calculate cluster centers in Degrees
        cluster_centers_deg = {}
        for i in range(k):
            cluster_vectors = X[labels == i]
            mean_vector = np.mean(cluster_vectors, axis=0)
            
            # Convert back to angle using arctan2
            center_rad = np.arctan2(mean_vector[1], mean_vector[0])
            center_deg = np.degrees(center_rad) % 360
            cluster_centers_deg[i] = center_deg
            
        return labels, cluster_centers_deg

    def get_closest_images(self, data_points, labels, cluster_centers):
        """
        For each cluster, finds the image whose center is closest to the cluster center.
        """
        results = {}
        unique_labels = np.unique(labels)
        
        for label in unique_labels:
            center_angle = cluster_centers[label]
            
            # Filter points belonging to this cluster
            cluster_indices = np.where(labels == label)[0]
            cluster_points = [data_points[i] for i in cluster_indices]
            
            best_image = None
            min_dist = float('inf')
            
            for p in cluster_points:
                img_center = p['image_center_deg']
                
                # Calculate circular angular distance
                diff = abs(center_angle - img_center)
                dist = min(diff, 360 - diff)
                
                if dist < min_dist:
                    min_dist = dist
                    best_image = p
            
            results[label] = {
                'cluster_center': center_angle,
                'best_image_id': best_image['image_id'],
                'best_image_center': best_image['image_center_deg'],
                'prediction_global_angle': best_image['global_angle']
            }
            
        return results

    def plot_results(self, data_points, labels, results):
        """
        Visualizes the points, clusters, and selected images on a polar plot.
        """
        fig = plt.figure(figsize=(8, 8))
        ax = fig.add_subplot(111, projection='polar')
        
        # 1. Plot all predicted points
        angles_rad = [np.deg2rad(d['global_angle']) for d in data_points]
        ax.scatter(angles_rad, [1]*len(angles_rad), c=labels, cmap='viridis', s=100, alpha=0.6, label='Predicted Points')
        
        # 2. Plot Cluster Centers
        for lbl, info in results.items():
            center_rad = np.deg2rad(info['cluster_center'])
            ax.plot([center_rad, center_rad], [0, 1.2], 'k--', lw=1) # Line to center
            ax.scatter(center_rad, 1.2, marker='X', s=200, c='red', zorder=10, label='Cluster Center' if lbl==0 else "")
            
            # 3. Plot Selected Image Center
            best_img_rad = np.deg2rad(info['best_image_center'])
            ax.scatter(best_img_rad, 0.9, marker='s', s=150, facecolors='none', edgecolors='red', linewidth=2, label='Selected Img Center' if lbl==0 else "")
            
        ax.set_yticks([])
        ax.set_title("Circular Clustering Results", va='bottom')
        ax.legend(loc='lower right', bbox_to_anchor=(1.3, 0))
        plt.show()


pipeline = PanoramaClusterer(global_width_px=4096, crop_fov_deg=80)

In [45]:
class PanoramaClusterer:
    def __init__(self, global_width_px=4096, crop_fov_deg=45):
        """
        global_width_px: The width of the full 360 equirectangular image.
        crop_fov_deg: The Field of View of your crops (e.g., 45 degrees).
        """
        self.global_width_px = global_width_px
        self.crop_fov_deg = crop_fov_deg
        
        # Calculate pixels per degree
        self.px_per_deg = global_width_px / 360.0
        self.crop_width_px = int(self.crop_fov_deg * self.px_per_deg)

    def map_local_to_global(self, image_center_deg, local_x_px):
        """
        Maps a local pixel coordinate (0 to crop_width) to a global angle (0-360).
        """
        # 1. Calculate offset from the center of the crop in pixels
        # Assuming local_x_px is 0-indexed relative to the crop
        crop_center_px = self.crop_width_px / 2.0
        offset_px = local_x_px - crop_center_px
        
        # 2. Convert pixel offset to degrees
        offset_deg = offset_px / self.px_per_deg
        
        # 3. Add to the image center angle and handle wrapping (modulo 360)
        global_angle = (image_center_deg + offset_deg) % 360
        return global_angle

    def cluster_points(self, data_points, k=3):
        """
        Clusters angular data by converting to Unit Vectors (sin, cos).
        """
        angles = np.array([d['global_angle'] for d in data_points])
        
        # Convert degrees to radians
        rads = np.deg2rad(angles)
        
        # Convert to Unit Vectors (Cartesian coordinates)
        # This solves the 0/360 boundary issue for K-Means
        X = np.column_stack([np.cos(rads), np.sin(rads)])
        
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X)
        
        # Calculate cluster centers in Degrees
        cluster_centers_deg = {}
        for i in range(k):
            cluster_vectors = X[labels == i]
            mean_vector = np.mean(cluster_vectors, axis=0)
            
            # Convert back to angle using arctan2
            center_rad = np.arctan2(mean_vector[1], mean_vector[0])
            center_deg = np.degrees(center_rad) % 360
            cluster_centers_deg[i] = center_deg
            
        return labels, cluster_centers_deg

    def get_closest_images(self, data_points, labels, cluster_centers):
        """
        For each cluster, finds the image whose center is closest to the cluster center.
        """
        results = {}
        unique_labels = np.unique(labels)
        
        for label in unique_labels:
            center_angle = cluster_centers[label]
            
            # Filter points belonging to this cluster
            cluster_indices = np.where(labels == label)[0]
            cluster_points = [data_points[i] for i in cluster_indices]
            
            best_image = None
            min_dist = float('inf')
            
            for p in cluster_points:
                img_center = p['image_center_deg']
                
                # Calculate circular angular distance
                diff = abs(center_angle - img_center)
                dist = min(diff, 360 - diff)
                
                if dist < min_dist:
                    min_dist = dist
                    best_image = p
            
            results[label] = {
                'cluster_center': center_angle,
                'best_image_id': best_image['image_id'],
                'best_image_center': best_image['image_center_deg'],
                'prediction_global_angle': best_image['global_angle']
            }
            
        return results

    def plot_results(self, data_points, labels, results):
        """
        Visualizes the points, clusters, and selected images on a polar plot.
        """
        fig = plt.figure(figsize=(8, 8))
        ax = fig.add_subplot(111, projection='polar')
        
        # 1. Plot all predicted points
        angles_rad = [np.deg2rad(d['global_angle']) for d in data_points]
        ax.scatter(angles_rad, [1]*len(angles_rad), c=labels, cmap='viridis', s=100, alpha=0.6, label='Predicted Points')
        
        # 2. Plot Cluster Centers
        for lbl, info in results.items():
            center_rad = np.deg2rad(info['cluster_center'])
            ax.plot([center_rad, center_rad], [0, 1.2], 'k--', lw=1) # Line to center
            ax.scatter(center_rad, 1.2, marker='X', s=200, c='red', zorder=10, label='Cluster Center' if lbl==0 else "")
            
            # 3. Plot Selected Image Center
            best_img_rad = np.deg2rad(info['best_image_center'])
            ax.scatter(best_img_rad, 0.9, marker='s', s=150, facecolors='none', edgecolors='red', linewidth=2, label='Selected Img Center' if lbl==0 else "")
            
        ax.set_yticks([])
        ax.set_title("Circular Clustering Results", va='bottom')
        ax.legend(loc='lower right', bbox_to_anchor=(1.3, 0))
        plt.show()

#### run

In [46]:
files = []


image_path = (f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_0.png")
files.append(image_path)
image_path = f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_20.png"
files.append(image_path)
image_path = f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_40.png"
files.append(image_path)
image_path = f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_60.png"
files.append(image_path)
image_path = f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_80.png"
files.append(image_path)
image_path = f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_100.png"
files.append(image_path)
image_path = f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_120.png"
files.append(image_path)
image_path = f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_140.png"
files.append(image_path)
image_path = f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_160.png"
files.append(image_path)
image_path = f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_180.png"
files.append(image_path)
image_path = f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_200.png"
files.append(image_path)
image_path = f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_220.png"
files.append(image_path)
image_path = f"examples/out2_cc6b8656-d52e-42d1-ab10-542588c9f9f0/img_cc6b8656-d52e-42d1-ab10-542588c9f9f0_240.png"
files.append(image_path)



In [47]:
for f in files:


    image = Image.open(image_path)

    # 2️⃣ Depth-Estimation Pipeline
    pipe = pipeline(task="depth-estimation", model="depth-anything/Depth-Anything-V2-Small-hf")

    # 3️⃣ Depth berechnen
    output = pipe(image)
    depth_map = output["depth"]

    # 4️⃣ Depth visualisieren
    depth_norm = (depth_map - np.min(depth_map)) / (np.max(depth_map) - np.min(depth_map))
    depth_vis = (depth_norm * 255).astype(np.uint8)

    # Depth-Map & Originalbild als NumPy
    depth_map_np = np.array(depth_map)
    orig_img_np = np.array(image.convert("RGB"))

    H, W = depth_map_np.shape
    orig_h, orig_w = orig_img_np.shape[:2]

    # # Streifen-Einstellungen (um mitte)
    # y_ratio = 0.5           # vertical position (0: up, 1: down)
    # strip_height_ratio = 0.25  # height of strip

    # Streifen-Einstellungen (boden bis mitte)
    y_ratio = 1           # vertical position (0: up, 1: down)
    strip_height_ratio = 1.2  # height of strip

    # Streifen in Depth-Map
    y_center = int(H * y_ratio)
    strip_h = max(5, int(H * strip_height_ratio))
    y_start = max(0, y_center - strip_h // 2)
    y_end = min(H, y_center + strip_h // 2)
    depth_strip = depth_map_np[y_start:y_end, :]

    # Streifen im Originalbild
    y_start_orig = int(orig_h * y_ratio - orig_h * strip_height_ratio / 2)
    y_end_orig = int(orig_h * y_ratio + orig_h * strip_height_ratio / 2)
    orig_strip = orig_img_np[y_start_orig:y_end_orig, :, :].copy()

    # Maximalpunkt im Streifen
    y_strip, x_strip = np.unravel_index(np.argmin(depth_strip, axis=None), depth_strip.shape)
    y_depth = y_start + y_strip
    x_depth = x_strip

    # Depth-Streifen für Visualisierung vorbereiten
    vis_map_strip = ((depth_strip - depth_strip.min()) / (depth_strip.max() - depth_strip.min()) * 255).astype(np.uint8)
    vis_depth_rgb_strip = cv2.cvtColor(vis_map_strip, cv2.COLOR_GRAY2RGB)

    # Marker auf Depth-Streifen (erstfundener Punkt)
    cv2.circle(vis_depth_rgb_strip, (x_strip, y_strip), radius=5, color=(255,0,0), thickness=-1)

    # Marker auf Originalbild (erstfundener Punkt)
    x_orig = int(x_depth * orig_w / W)
    y_in_strip_orig = int((y_strip / strip_h) * (y_end_orig - y_start_orig)) + y_start_orig
    orig_img_marked = orig_img_np.copy()
    cv2.circle(orig_img_marked, (x_orig, y_in_strip_orig), radius=5, color=(255,0,0), thickness=-1)

    # Streifen als Rechteck markieren
    cv2.rectangle(orig_img_marked, (0, y_start_orig), (orig_w-1, y_end_orig), color=(255,255,0), thickness=1)






    # Alle Punkte mit gleicher Tiefe (Toleranz für Float)
    tolerance = 1e-5
    ys, xs = np.where(np.abs(depth_strip - depth_strip[y_strip, x_strip]) < tolerance)
    num_points = len(xs)
    print(f"Anzahl der Punkte mit maximaler Tiefe im Streifen: {num_points}")

    # Mittelpunkt berechnen
    center_x = int(np.mean(xs))
    center_y = int(np.mean(ys))
    print(f"Mittelpunkt aller Punkte im Streifen-Koordinatensystem: (x={center_x}, y={center_y})")

    # Originalbild kopieren
    orig_img_marked = orig_img_np.copy()

    # Alle Punkte auf Originalbild markieren (blau)
    for y_pt, x_pt in zip(ys, xs):
        x_orig = int(x_pt * orig_w / W)
        y_orig = int((y_pt / strip_h) * (y_end_orig - y_start_orig)) + y_start_orig
        cv2.circle(orig_img_marked, (x_orig, y_orig), radius=3, color=(0,0,255), thickness=-1)

    # Erstfundener Punkt (rot)
    x_first_orig = int(x_strip * orig_w / W)
    y_first_orig = int((y_strip / strip_h) * (y_end_orig - y_start_orig)) + y_start_orig
    cv2.circle(orig_img_marked, (x_first_orig, y_first_orig), radius=5, color=(255,0,0), thickness=-1)

    # Mittelpunkt (grün)
    x_center_orig = int(center_x * orig_w / W)
    y_center_orig = int((center_y / strip_h) * (y_end_orig - y_start_orig)) + y_start_orig
    cv2.circle(orig_img_marked, (x_center_orig, y_center_orig), radius=5, color=(0,255,0), thickness=-1)

    # Streifen als Rechteck markieren
    cv2.rectangle(orig_img_marked, (0, y_start_orig), (orig_w-1, y_end_orig), color=(255,255,0), thickness=1)




    center_x_img = orig_w // 2
    center_y_img = orig_h // 2

    # Mittelpunkt der maximalen Tiefe (bereits in x_orig, y_orig)
    max_depth_point = np.array([x_orig, y_orig])
    image_center = np.array([center_x_img, center_y_img])

TypeError: 'PanoramaClusterer' object is not callable

In [ ]:
processed_points = []
for img_id, img_center, local_x in raw_predictions:
    g_angle = pipeline.map_local_to_global(img_center, local_x)
    processed_points.append({
        'image_id': img_id,
        'image_center_deg': img_center,
        'local_x': local_x,
        'global_angle': g_angle
    })

print("Mapped Angles:")
for p in processed_points:
    print(f"ID: {p['image_id']:<10} Center: {p['image_center_deg']:<5} -> Global: {p['global_angle']:.2f}°")


labels, centers = pipeline.cluster_points(processed_points, k=3)


best_images = pipeline.get_closest_images(processed_points, labels, centers)


print("\n" + "="40)
print("FINAL RESULTS")
print("="40)
for cluster_id, data in best_images.items():
    print(f"Cluster {cluster_id}")
    print(f"  Cluster Center: {data['cluster_center']:.2f}°")
    print(f"  Selected Image: {data['best_image_id']} (Center: {data['best_image_center']}°)")
    print("-" * 20)